# 043: Python Dictionaries — Complete A–Z Method Reference, Hash Table Internals, and CPython Evolution

## 1. WHAT IS A PYTHON DICTIONARY?

A Python `dict` is an **ordered (since CPython 3.7), mutable mapping** of **hashable keys** to arbitrary values. It is the single most important data structure in Python — classes, modules, namespaces, keyword arguments, and global/local scopes are all implemented as dictionaries.

### CPython Internal Representation (Compact Dict — CPython 3.6+)

Before CPython 3.6, dictionaries used a single large hash table where each entry stored the hash, key, and value. This wasted memory because the table was typically only 1/3 to 2/3 full.

**Compact Dict (Raymond Hettinger's design, CPython 3.6+)** splits the structure into two arrays:
1. **Indices array** (`dk_indices`): A dense array of integer indices into the entries array. Sized to a power of 2. Each slot holds either an index into `dk_entries`, or a sentinel (`DKIX_EMPTY = -1`, `DKIX_DUMMY = -2`).
2. **Entries array** (`dk_entries`): A compact array of `(hash, key, value)` tuples, stored in **insertion order**. No gaps.

```
Indices: [  -1 |  0 | -1 |  1 | -1 | -1 |  2 | -1 ]  (hash table, power-of-2 size)
Entries: [ (hash_a, key_a, val_a),                      (insertion-ordered, compact)
           (hash_b, key_b, val_b),
           (hash_c, key_c, val_c) ]
```

### Why This Design?
- **Memory savings**: ~25-35% less memory than the old design because the indices array uses 1-byte or 2-byte integers for small dicts instead of full 24-byte entries.
- **Insertion order preservation**: Entries are stored in insertion order in the entries array, giving `dict` guaranteed ordering as a language specification (Python 3.7+).
- **Cache-friendly iteration**: Iterating over `dk_entries` is a sequential memory scan with no gaps.

### Hash Collision Resolution
- Same as sets: **open addressing** with **perturbation-based probing**:
  $$j = (5j + 1 + \text{perturb}) \bmod \text{table\_size}; \quad \text{perturb} \mathrel{>>=} 5$$

### Resizing Strategy
- When the dict is more than 2/3 full (`used * 3 >= capacity * 2`), CPython resizes the indices array to the next power of 2.
- For dicts that had many deletions, CPython may resize to the **same** size (compacting the entries array to remove dummy entries).

---


In [ ]:
import sys

print("=" * 70)
print("SECTION 1: Dictionary Memory Layout & Growth")
print("=" * 70)

d = {}
prev = sys.getsizeof(d)
print(f"{'Keys':>6} | {'Bytes':>6} | Event")
print("-" * 40)
print(f"{'0':>6} | {prev:>6} | Empty dict")

for i in range(50):
    d[i] = i * i
    curr = sys.getsizeof(d)
    if curr != prev:
        print(f"{len(d):>6} | {curr:>6} | RESIZE triggered")
        prev = curr



---
## 2. COMPLETE A–Z METHOD REFERENCE

---
### 2.1 `dict.get(key[, default])`
- **What it does**: Returns the value for `key` if it exists, otherwise returns `default` (default is `None`).
- **Returns**: The value or default.
- **Does NOT raise** `KeyError`.
- **Time**: $O(1)$ average.
- **Use case**: Safe access when you're not sure if a key exists.


In [ ]:
print("\n" + "=" * 70)
print("METHOD: dict.get(key, default)")
print("=" * 70)

config = {"host": "localhost", "port": 8080, "debug": True}

print(f"get('host'):                 {config.get('host')}")
print(f"get('timeout'):              {config.get('timeout')}")          # None
print(f"get('timeout', 30):          {config.get('timeout', 30)}")     # Custom default

# Comparison: get() vs [] access
try:
    _ = config["timeout"]
except KeyError as e:
    print(f"config['timeout']:           KeyError: {e}")

# Edge case: key exists with value None
d = {"key": None}
print(f"get('key') when val is None: {d.get('key')}")   # None (exists!)
print(f"get('missing'):              {d.get('missing')}")  # None (default)
# These are indistinguishable! Use 'key in d' to check existence.



### 2.2 `dict.setdefault(key[, default])`
- **What it does**: If `key` is in the dict, return its value. If not, insert `key` with value `default` and return `default`.
- **Returns**: The value for `key`.
- **Time**: $O(1)$ average.
- **Use case**: Initializing dict entries on first access (grouping pattern).


In [ ]:
print("\n" + "=" * 70)
print("METHOD: dict.setdefault(key, default)")
print("=" * 70)

# Grouping pattern: group words by first letter
words = ["apple", "banana", "avocado", "blueberry", "cherry", "apricot"]
groups = {}
for word in words:
    groups.setdefault(word[0], []).append(word)
print(f"Grouped by first letter:     {groups}")

# setdefault does NOT overwrite existing values
d = {"x": 10}
result = d.setdefault("x", 999)
print(f"setdefault on existing key:  {result} (original preserved)")



### 2.3 `dict.update([other])`
- **What it does**: Updates the dict with key/value pairs from `other` (dict, iterable of key-value pairs, or keyword args). Existing keys are overwritten.
- **Returns**: `None`.
- **Time**: $O(k)$ where $k$ is the number of items in `other`.


In [ ]:
print("\n" + "=" * 70)
print("METHOD: dict.update()")
print("=" * 70)

base = {"a": 1, "b": 2}

# Update from another dict
base.update({"b": 20, "c": 30})
print(f"update(dict):                {base}")

# Update from keyword arguments
base.update(d=40, e=50)
print(f"update(kwargs):              {base}")

# Update from list of tuples
base.update([("f", 60), ("g", 70)])
print(f"update(list of tuples):      {base}")



### 2.4 `dict.pop(key[, default])`
- **What it does**: Removes `key` and returns its value. If `key` is not found, returns `default` or raises `KeyError`.
- **Time**: $O(1)$ average.


In [ ]:
print("\n" + "=" * 70)
print("METHOD: dict.pop(key, default)")
print("=" * 70)

d = {"a": 1, "b": 2, "c": 3}
val = d.pop("b")
print(f"pop('b') returned:           {val}, dict now: {d}")

# With default (no error if missing)
val = d.pop("z", "NOT_FOUND")
print(f"pop('z', default):           {val}")

# Without default — raises KeyError
try:
    d.pop("z")
except KeyError as e:
    print(f"pop('z') no default:         KeyError: {e}")



### 2.5 `dict.popitem()`
- **What it does**: Removes and returns the **last inserted** key-value pair as a tuple (LIFO order, since Python 3.7).
- **Raises**: `KeyError` if the dict is empty.
- **Time**: $O(1)$.


In [ ]:
print("\n" + "=" * 70)
print("METHOD: dict.popitem()")
print("=" * 70)

d = {"first": 1, "second": 2, "third": 3}
item = d.popitem()
print(f"popitem() returned:          {item}, dict now: {d}")
item = d.popitem()
print(f"popitem() returned:          {item}, dict now: {d}")

try:
    {}.popitem()
except KeyError as e:
    print(f"popitem() on empty dict:     KeyError: {e}")



### 2.6 `dict.keys()`, `dict.values()`, `dict.items()`
- Return **view objects** — dynamic views that reflect changes to the dict in real time.
- Views support set operations (`&`, `|`, `-`, `^`) for `keys()` and `items()` views (but NOT `values()` because values aren't guaranteed unique or hashable).
- **Time**: Creating a view is $O(1)$. Iterating is $O(n)$.


In [ ]:
print("\n" + "=" * 70)
print("METHOD: dict.keys(), dict.values(), dict.items()")
print("=" * 70)

d = {"a": 1, "b": 2, "c": 3}

# View objects are dynamic
keys_view = d.keys()
print(f"keys view:                   {keys_view}")
d["d"] = 4
print(f"keys view after mutation:    {keys_view}")  # Automatically updated!

# Set operations on keys view
d1 = {"a": 1, "b": 2, "c": 3}
d2 = {"b": 20, "c": 30, "d": 40}
print(f"Common keys (d1 & d2):       {d1.keys() & d2.keys()}")
print(f"All keys (d1 | d2):          {d1.keys() | d2.keys()}")
print(f"Only in d1 (d1 - d2):        {d1.keys() - d2.keys()}")

# Items view supports set ops if values are hashable
print(f"Common items:                {d1.items() & d2.items()}")



### 2.7 `dict.copy()`
- Returns a **shallow copy**. $O(n)$.
- Nested mutable values are NOT deep-copied.


In [ ]:
print("\n" + "=" * 70)
print("METHOD: dict.copy() — Shallow vs Deep")
print("=" * 70)

import copy

original = {"a": [1, 2], "b": [3, 4]}

shallow = original.copy()
shallow["a"].append(99)
print(f"Original after shallow mut:  {original}")  # ALSO mutated!

original = {"a": [1, 2], "b": [3, 4]}
deep = copy.deepcopy(original)
deep["a"].append(99)
print(f"Original after deep mut:     {original}")  # NOT mutated



### 2.8 `dict.clear()`
- Removes all items. $O(n)$.


In [ ]:
print("\n" + "=" * 70)
print("METHOD: dict.clear()")
print("=" * 70)

d = {"a": 1, "b": 2}
alias = d
d.clear()
print(f"d after clear:               {d}")
print(f"alias after clear:           {alias}")  # Also empty (same object)



### 2.9 `dict.fromkeys(iterable[, value])`
- **Class method** that creates a new dict with keys from `iterable`, all set to `value` (default `None`).
- **Warning**: If `value` is mutable, ALL keys share the SAME object!


In [ ]:
print("\n" + "=" * 70)
print("METHOD: dict.fromkeys(iterable, value)")
print("=" * 70)

d = dict.fromkeys(["a", "b", "c"], 0)
print(f"fromkeys with int:           {d}")

# The mutable value trap
d = dict.fromkeys(["x", "y", "z"], [])
d["x"].append(1)
print(f"fromkeys with list trap:     {d}")
# ALL values are the SAME list object! Mutating one mutates all.

# Safe alternative
d = {k: [] for k in ["x", "y", "z"]}
d["x"].append(1)
print(f"Comprehension (safe):        {d}")



---
## 3. DICT COMPREHENSIONS


In [ ]:
print("\n" + "=" * 70)
print("SECTION 3: Dict Comprehensions")
print("=" * 70)

# Basic comprehension
squares = {x: x*x for x in range(6)}
print(f"Squares dict:                {squares}")

# Conditional comprehension
even_squares = {x: x*x for x in range(10) if x % 2 == 0}
print(f"Even squares:                {even_squares}")

# Inverting a dict (swap keys and values)
original = {"a": 1, "b": 2, "c": 3}
inverted = {v: k for k, v in original.items()}
print(f"Inverted dict:               {inverted}")

# Merging two dicts with comprehension (Python 3.9+ has |)
d1 = {"a": 1, "b": 2}
d2 = {"b": 20, "c": 30}
merged = {**d1, **d2}  # d2 values overwrite d1
print(f"Merged (unpacking):          {merged}")

# Python 3.9+ merge operator
# merged = d1 | d2  # Same result



---
## 4. INSERTION ORDER PRESERVATION

- **CPython 3.6**: Insertion order preservation was an implementation detail.
- **Python 3.7**: Insertion order preservation became part of the **language specification**.
- `dict.popitem()` now removes the LAST inserted item (LIFO).
- Iteration order matches insertion order.


In [ ]:
print("\n" + "=" * 70)
print("SECTION 4: Insertion Order Preservation")
print("=" * 70)

d = {}
d["third"] = 3
d["first"] = 1
d["second"] = 2
print(f"Insertion order:             {list(d.keys())}")

# Updating an existing key does NOT change its position
d["third"] = 33
print(f"After updating 'third':      {list(d.keys())}")  # Order unchanged

# Deleting and re-inserting DOES change position
del d["first"]
d["first"] = 1
print(f"After del+re-insert 'first': {list(d.keys())}")  # 'first' now at end



---
## 5. `__missing__` AND `collections.defaultdict`

- Subclassing `dict` and overriding `__missing__(self, key)` lets you customize what happens when a key is not found.
- `collections.defaultdict` uses this pattern: it calls a factory function to create a default value for missing keys.


In [ ]:
print("\n" + "=" * 70)
print("SECTION 5: __missing__ and defaultdict")
print("=" * 70)

from collections import defaultdict

# defaultdict with int factory (default value = 0)
word_count = defaultdict(int)
for word in "the cat sat on the mat the cat".split():
    word_count[word] += 1
print(f"Word counts:                 {dict(word_count)}")

# defaultdict with list factory
grouped = defaultdict(list)
pairs = [("fruit", "apple"), ("vegetable", "carrot"), ("fruit", "banana")]
for category, item in pairs:
    grouped[category].append(item)
print(f"Grouped:                     {dict(grouped)}")

# Custom __missing__
class AutoKeyDict(dict):
    def __missing__(self, key):
        self[key] = f"auto_{key}"
        return self[key]

auto = AutoKeyDict()
print(f"auto['x']:                   {auto['x']}")
print(f"auto after access:           {auto}")



---
## 6. DICT vs OBJECT ATTRIBUTE MAPPING

- Python objects store their instance attributes in a `__dict__` dictionary (unless `__slots__` is defined).
- `getattr(obj, 'x')` is equivalent to `obj.__dict__['x']` for simple cases.
- Classes, modules, and functions all have `__dict__` attributes.


In [ ]:
print("\n" + "=" * 70)
print("SECTION 6: Dict as Object Internals")
print("=" * 70)

class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

p = Person("Alice", 30)
print(f"p.__dict__:                  {p.__dict__}")

# Modifying __dict__ directly modifies the object
p.__dict__["email"] = "alice@example.com"
print(f"p.email:                     {p.email}")

# Module __dict__
import math
print(f"math.__dict__ keys (first 5): {list(math.__dict__.keys())[:5]}")



---
## 7. PERFORMANCE BENCHMARKS


In [ ]:
print("\n" + "=" * 70)
print("SECTION 7: Dict Performance Benchmarks")
print("=" * 70)

import time

N = 100_000

# Insertion speed
t0 = time.perf_counter()
d = {}
for i in range(N):
    d[i] = i
t1 = time.perf_counter()
print(f"Insert {N:,} items:           {(t1-t0)*1000:.1f} ms")

# Lookup speed
t0 = time.perf_counter()
for i in range(N):
    _ = d[i]
t1 = time.perf_counter()
print(f"Lookup {N:,} items:           {(t1-t0)*1000:.1f} ms")

# Deletion speed
t0 = time.perf_counter()
for i in range(N):
    del d[i]
t1 = time.perf_counter()
print(f"Delete {N:,} items:           {(t1-t0)*1000:.1f} ms")

# Membership test
d = dict.fromkeys(range(N))
t0 = time.perf_counter()
for i in range(N):
    _ = i in d
t1 = time.perf_counter()
print(f"Membership {N:,} tests:       {(t1-t0)*1000:.1f} ms")



---
## 8. FAANG & RESEARCH INTERVIEW QUESTIONS

### Q1: What is the average time complexity of dict lookup, insertion, and deletion?
**A**: All are $O(1)$ average, $O(n)$ worst case (when all keys hash to the same slot). Python's perturbation-based probing makes worst-case extremely unlikely with well-distributed hashes.

### Q2: Why are Python dicts ordered since 3.7?
**A**: The compact dict implementation (CPython 3.6) stores entries in a separate insertion-ordered array. Since entries are appended in order and never rearranged (deletions leave tombstones), iteration reflects insertion order. Python 3.7 made this a language guarantee.

### Q3: What happens when you use a mutable default value in `dict.fromkeys()`?
**A**: All keys share the **same** mutable object. Mutating through one key affects all keys. Use a dict comprehension `{k: [] for k in keys}` instead to create independent objects.

### Q4: How does CPython resize a dict?
**A**: When the indices table is more than 2/3 full, CPython allocates a new indices table at the next power-of-2 size and re-inserts all entries. The entries array is compacted to remove dummy (deleted) entries. This takes $O(n)$ but happens infrequently enough that insertion remains $O(1)$ amortized.

### Q5: What is the `__missing__` method?
**A**: When `dict.__getitem__` fails to find a key, it checks if the dict subclass defines `__missing__`. If so, it calls `self.__missing__(key)` instead of raising `KeyError`. `collections.defaultdict` uses this mechanism: its `__missing__` calls the factory function, stores the result, and returns it.
